# AnimeLoom — Character Training & Animation Pipeline (Google Colab)

Train character LoRAs for consistent anime identity across long-form animation.
Supports **SDXL** (Animagine XL 3.1) for keyframes and **SD 1.5** (DreamShaper 8) for AnimateDiff video.

**What this notebook does:**
1. Sets up AnimeLoom environment on Colab
2. Mounts Google Drive for persistent storage
3. Downloads datasets or uploads your own character images
4. Auto-captions images with BLIP
5. Trains character LoRAs (SDXL + SD 1.5 for AnimateDiff)
6. Tests trained characters for consistency
7. Exports everything for AnimeLoom's pipeline
8. **Generates animated video** — single clips or 2+ minute long videos

| Colab Tier | GPU | Session Limit | Good For |
|------------|-----|---------------|----------|
| Free | T4 (15GB) | ~90 min | 1–2 characters, short clips |
| Pro ($10) | T4/A100 | ~12 hours | Full training + 2-min video generation |

---

## 1. Environment Setup
Clones AnimeLoom, installs dependencies, mounts Google Drive.

In [ ]:
#@title 1. Setup AnimeLoom Environment
#@markdown Clones the repo, installs deps, and mounts Google Drive.

import os, sys, subprocess
from pathlib import Path

# --- Clone AnimeLoom ---
REPO_URL = "https://github.com/JoelJohnsonThomas/AnimeLoom.git"  #@param {type:"string"}
REPO_DIR = Path("/content/AnimeLoom")

if not REPO_DIR.exists():
    print("Cloning AnimeLoom…")
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    print(f"AnimeLoom already at {REPO_DIR}")

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))

# --- Mount Google Drive ---
from google.colab import drive
drive.mount("/content/drive")

# --- Warehouse on Drive (persists across sessions) ---
WAREHOUSE = Path("/content/drive/MyDrive/AniLoom/warehouse")
WAREHOUSE.mkdir(parents=True, exist_ok=True)
os.environ["AI_CACHE_ROOT"] = str(WAREHOUSE)

for sub in ["lora", "models", "datasets/raw", "datasets/tagged", "outputs", "checkpoints"]:
    (WAREHOUSE / sub).mkdir(parents=True, exist_ok=True)

print(f"Warehouse: {WAREHOUSE}")

# --- Install dependencies ---
print("\nInstalling dependencies…")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "torch", "torchvision", "--index-url", "https://download.pytorch.org/whl/cu118"],
    capture_output=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "diffusers>=0.24.0", "transformers>=4.35.0", "accelerate", "safetensors",
     "peft", "bitsandbytes", "xformers",
     "opencv-python-headless", "pillow", "numpy", "scikit-image",
     "datasets", "huggingface-hub", "tqdm",
     "controlnet-aux>=0.0.7",
     "optimum-quanto"],
    capture_output=True,
)

import torch
print(f"\nPyTorch {torch.__version__}  CUDA {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}  VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

print("\n✅ Setup complete!")

## 2. Get Character Images
Choose **one** of the three options below.

In [ ]:
#@title 2a. Download from HuggingFace (deepghs/character_similarity)
#@markdown Downloads pre-organised anime characters with multiple images each.

SPLIT = "v0_tiny"  #@param ["v0_tiny", "v1_pruned", "v1"]
MAX_CHARACTERS = 10  #@param {type:"slider", min:5, max:200, step:5}

from scripts.prepare_dataset import download_huggingface
download_huggingface(split=SPLIT, max_characters=MAX_CHARACTERS)

print(f"\nCharacter folders in {WAREHOUSE / 'datasets/raw'}:")
for d in sorted((WAREHOUSE / 'datasets/raw').iterdir()):
    if d.is_dir():
        n = len([f for f in d.iterdir() if f.suffix.lower() in {'.png','.jpg','.jpeg','.webp'}])
        print(f"  {d.name}: {n} images")

In [ ]:
#@title 2b. Upload your own images
#@markdown Upload images to a named character folder.

CHARACTER_NAME = "yuki"  #@param {type:"string"}

from google.colab import files
from pathlib import Path
import shutil, os

WAREHOUSE = Path(os.environ["AI_CACHE_ROOT"])
char_dir = WAREHOUSE / "datasets" / "raw" / CHARACTER_NAME.lower().replace(" ", "_")
char_dir.mkdir(parents=True, exist_ok=True)

print(f"Select 5–20 images of '{CHARACTER_NAME}' (different poses/expressions):")
uploaded = files.upload()

for fname, data in uploaded.items():
    dst = char_dir / fname
    dst.write_bytes(data)
    print(f"  Saved {fname} → {dst}")

total = len(list(char_dir.glob("*")))
print(f"\n✅ {total} images in {char_dir}")
if total < 5:
    print("⚠️  Recommend at least 5 images for good LoRA quality.")

In [ ]:
#@title 2c. Import from a Google Drive folder
#@markdown Point to a Drive folder containing character images.

CHARACTER_NAME = "yuki"  #@param {type:"string"}
DRIVE_FOLDER = "/content/drive/MyDrive/my_characters/yuki"  #@param {type:"string"}

from scripts.prepare_dataset import import_local
import_local(DRIVE_FOLDER, character_name=CHARACTER_NAME)

## 3. Auto-Caption Images with BLIP
Generates text captions for each image. Captions help the LoRA learn *what* it's seeing.

In [ ]:
#@title 3. Auto-Caption with BLIP
#@markdown Select which characters to caption (comma-separated), or "all".

CHARACTERS = "all"  #@param {type:"string"}

import os
from pathlib import Path
from scripts.prepare_dataset import caption_folder

WAREHOUSE = Path(os.environ["AI_CACHE_ROOT"])
raw_dir = WAREHOUSE / "datasets" / "raw"

if CHARACTERS.strip().lower() == "all":
    char_list = [d for d in sorted(raw_dir.iterdir()) if d.is_dir()]
else:
    char_list = [raw_dir / c.strip() for c in CHARACTERS.split(",")]

print(f"Captioning {len(char_list)} character(s)…\n")
for char_dir in char_list:
    if not char_dir.exists():
        print(f"⚠️  Skipping {char_dir.name} — folder not found")
        continue
    caption_folder(str(char_dir))
    print()

print("✅ Captioning complete!")

## 4. Train Character LoRA
This is the main training cell. Uses AnimeLoom's `scripts/train_lora.py` under the hood.

In [ ]:
#@title 4. Train a Single Character LoRA
#@markdown Configure training parameters and run.

CHARACTER_NAME = "yuki"  #@param {type:"string"}
USE_CAPTIONS = True  #@param {type:"boolean"}
LORA_RANK = 32  #@param {type:"slider", min:8, max:64, step:8}
LEARNING_RATE = 1e-4  #@param {type:"number"}
TRAIN_STEPS = 1000  #@param {type:"slider", min:200, max:3000, step:100}
BATCH_SIZE = 1  #@param {type:"slider", min:1, max:4, step:1}
RESOLUTION = 1024  #@param {type:"slider", min:256, max:1024, step:128}
BASE_MODEL = "cagliostrolab/animagine-xl-3.1"  #@param ["cagliostrolab/animagine-xl-3.1", "stabilityai/stable-diffusion-xl-base-1.0", "sd2-community/stable-diffusion-2-1"]

import os, torch
from pathlib import Path
from scripts.train_lora import train

WAREHOUSE = Path(os.environ["AI_CACHE_ROOT"])
char_id = CHARACTER_NAME.lower().replace(" ", "_")

if USE_CAPTIONS:
    image_dir = WAREHOUSE / "datasets" / "tagged" / char_id
else:
    image_dir = WAREHOUSE / "datasets" / "raw" / char_id

if not image_dir.exists():
    raise FileNotFoundError(
        f"No images at {image_dir}. "
        f"Run step 2 to get images, then step 3 to caption them."
    )

print(f"GPU: {torch.cuda.get_device_name(0)}" if torch.cuda.is_available() else "⚠️ No GPU!")
print(f"Training '{CHARACTER_NAME}' from {image_dir}")
print(f"Rank={LORA_RANK}  LR={LEARNING_RATE}  Steps={TRAIN_STEPS}  Res={RESOLUTION}")
print("=" * 60)

lora_dir = train(
    character_name=CHARACTER_NAME,
    image_dir=image_dir,
    rank=LORA_RANK,
    lr=LEARNING_RATE,
    steps=TRAIN_STEPS,
    batch_size=BATCH_SIZE,
    resolution=RESOLUTION,
    base_model=BASE_MODEL,
    use_captions=USE_CAPTIONS,
)

print(f"\n✅ LoRA saved to: {lora_dir}")
print("This persists on Google Drive — available in future sessions.")

In [ ]:
#@title 4b. Batch Train Multiple Characters
#@markdown Train several characters sequentially.

CHARACTER_NAMES = "yuki, kenji, sakura"  #@param {type:"string"}
USE_CAPTIONS = True  #@param {type:"boolean"}
LORA_RANK = 32  #@param {type:"slider", min:8, max:64, step:8}
LEARNING_RATE = 1e-4  #@param {type:"number"}
TRAIN_STEPS = 800  #@param {type:"slider", min:200, max:3000, step:100}
BATCH_SIZE = 1  #@param {type:"slider", min:1, max:4, step:1}
RESOLUTION = 1024  #@param {type:"slider", min:256, max:1024, step:128}
BASE_MODEL = "cagliostrolab/animagine-xl-3.1"  #@param ["cagliostrolab/animagine-xl-3.1", "stabilityai/stable-diffusion-xl-base-1.0", "sd2-community/stable-diffusion-2-1"]

import os, gc, torch
from pathlib import Path
from scripts.train_lora import train

WAREHOUSE = Path(os.environ["AI_CACHE_ROOT"])
char_list = [c.strip() for c in CHARACTER_NAMES.split(",")]

results = {}
for i, name in enumerate(char_list, 1):
    char_id = name.lower().replace(" ", "_")
    subdir = "tagged" if USE_CAPTIONS else "raw"
    image_dir = WAREHOUSE / "datasets" / subdir / char_id

    if not image_dir.exists():
        print(f"\n⚠️  [{i}/{len(char_list)}] Skipping '{name}' — no images at {image_dir}")
        results[name] = "SKIPPED"
        continue

    print(f"\n{'='*60}")
    print(f"[{i}/{len(char_list)}] Training '{name}'…")
    print(f"{'='*60}")

    try:
        lora_dir = train(
            character_name=name,
            image_dir=image_dir,
            rank=LORA_RANK,
            lr=LEARNING_RATE,
            steps=TRAIN_STEPS,
            batch_size=BATCH_SIZE,
            resolution=RESOLUTION,
            base_model=BASE_MODEL,
            use_captions=USE_CAPTIONS,
        )
        results[name] = f"OK → {lora_dir}"
    except Exception as e:
        print(f"❌ Failed: {e}")
        results[name] = f"FAILED: {e}"

    # Free VRAM between characters
    gc.collect()
    torch.cuda.empty_cache()

print(f"\n{'='*60}")
print("Batch training results:")
for name, status in results.items():
    print(f"  {name}: {status}")

## 4c. Train SD 1.5 LoRA for AnimateDiff
Your SDXL LoRAs won't work with AnimateDiff (SD 1.5). Train a separate SD 1.5 LoRA using **DreamShaper 8** as the base model. This is required for animated video generation.

In [ ]:
#@title 4c. Train SD 1.5 LoRA for AnimateDiff
#@markdown Trains a separate SD 1.5 LoRA using DreamShaper 8 (anime SD 1.5 model).
#@markdown Required for AnimateDiff video generation. Saves to `warehouse/lora/{name}_sd15/`.

CHARACTER_NAMES = "yuki_nagato, denji, sakura_haruno"  #@param {type:"string"}
USE_CAPTIONS = True  #@param {type:"boolean"}
LORA_RANK = 32  #@param {type:"slider", min:8, max:64, step:8}
LEARNING_RATE = 1e-4  #@param {type:"number"}
TRAIN_STEPS = 800  #@param {type:"slider", min:200, max:3000, step:100}
BATCH_SIZE = 1  #@param {type:"slider", min:1, max:4, step:1}
BASE_MODEL = "Lykon/dreamshaper-8"  #@param ["Lykon/dreamshaper-8", "Lykon/dreamshaper-7", "runwayml/stable-diffusion-v1-5"]

import os, gc, torch
from pathlib import Path
from scripts.train_lora import train

WAREHOUSE = Path(os.environ["AI_CACHE_ROOT"])
char_list = [c.strip() for c in CHARACTER_NAMES.split(",")]

print(f"Training SD 1.5 LoRAs for AnimateDiff compatibility")
print(f"Base model: {BASE_MODEL}")
print(f"Resolution: 512 (auto-set for SD 1.5)")
print(f"Characters: {char_list}")
print("=" * 60)

results = {}
for i, name in enumerate(char_list, 1):
    char_id = name.lower().replace(" ", "_")
    subdir = "tagged" if USE_CAPTIONS else "raw"
    image_dir = WAREHOUSE / "datasets" / subdir / char_id

    if not image_dir.exists():
        print(f"\n⚠️  [{i}/{len(char_list)}] Skipping '{name}' — no images at {image_dir}")
        results[name] = "SKIPPED"
        continue

    # Check if SD 1.5 LoRA already exists
    sd15_dir = WAREHOUSE / "lora" / f"{char_id}_sd15"
    if (sd15_dir / "adapter_model.safetensors").exists() or (sd15_dir / "pytorch_lora_weights.safetensors").exists():
        print(f"\n[{i}/{len(char_list)}] '{name}' already has SD 1.5 LoRA at {sd15_dir} — skipping")
        results[name] = f"EXISTS → {sd15_dir}"
        continue

    print(f"\n{'='*60}")
    print(f"[{i}/{len(char_list)}] Training SD 1.5 LoRA for '{name}'…")
    print(f"{'='*60}")

    try:
        lora_dir = train(
            character_name=name,
            image_dir=image_dir,
            rank=LORA_RANK,
            lr=LEARNING_RATE,
            steps=TRAIN_STEPS,
            batch_size=BATCH_SIZE,
            resolution=512,
            base_model=BASE_MODEL,
            use_captions=USE_CAPTIONS,
        )
        results[name] = f"OK → {lora_dir}"

        # Register SD 1.5 LoRA in memory bank
        from director.memory_bank import AssetMemoryBank
        memory = AssetMemoryBank(str(WAREHOUSE))
        lora_path = lora_dir / "pytorch_lora_weights.safetensors"
        if not lora_path.exists():
            lora_path = lora_dir / "adapter_model.safetensors"
        memory.update_character_lora(name, str(lora_path), model_type="sd15")
        print(f"  Registered SD 1.5 LoRA in memory bank")

    except Exception as e:
        print(f"❌ Failed: {e}")
        results[name] = f"FAILED: {e}"

    gc.collect()
    torch.cuda.empty_cache()

print(f"\n{'='*60}")
print("SD 1.5 LoRA training results:")
for name, status in results.items():
    print(f"  {name}: {status}")
print(f"\nSD 1.5 LoRAs saved to: warehouse/lora/{{name}}_sd15/")

## 5. Test Trained Character
Generate sample images with the trained LoRA to verify consistency.

In [ ]:
#@title 5. Test Character LoRA
#@markdown Generate images to verify the trained character looks consistent.

CHARACTER_NAME = "yuki_nagato"  #@param {type:"string"}
TEST_PROMPT = "yuki_nagato, 1girl, smiling, looking at viewer, anime portrait, detailed, masterpiece"  #@param {type:"string"}
NEGATIVE_PROMPT = "blurry, low quality, deformed, extra fingers, worst quality"  #@param {type:"string"}
NUM_IMAGES = 4  #@param {type:"slider", min:1, max:8, step:1}
GUIDANCE_SCALE = 7.5  #@param {type:"slider", min:1.0, max:15.0, step:0.5}
INFERENCE_STEPS = 30  #@param {type:"slider", min:10, max:50, step:5}
IMAGE_SIZE = 768  #@param {type:"slider", min:512, max:1024, step:128}

import os, json, gc, torch
from pathlib import Path
from diffusers import DPMSolverMultistepScheduler
import matplotlib.pyplot as plt

WAREHOUSE = Path(os.environ["AI_CACHE_ROOT"])
char_id = CHARACTER_NAME.lower().replace(" ", "_")
lora_dir = WAREHOUSE / "lora" / char_id

# Load metadata
meta_path = lora_dir / "metadata.json"
if not meta_path.exists():
    raise FileNotFoundError(f"No trained LoRA for '{CHARACTER_NAME}' at {lora_dir}")

meta = json.loads(meta_path.read_text())
is_sdxl = meta.get("is_sdxl", "xl" in meta.get("base_model", "").lower() or "animagine" in meta.get("base_model", "").lower())

print(f"Character: {meta['character_name']}")
print(f"Base model: {meta['base_model']} ({'SDXL' if is_sdxl else 'SD'})")
print(f"Rank: {meta['rank']}  Steps: {meta['training_steps']}  Images: {meta['num_images']}")

# Free VRAM from any prior work
gc.collect()
torch.cuda.empty_cache()

# Load pipeline
print("\nLoading model…")
if is_sdxl:
    from diffusers import StableDiffusionXLPipeline
    pipe = StableDiffusionXLPipeline.from_pretrained(
        meta["base_model"],
        torch_dtype=torch.float16,
    )
else:
    from diffusers import StableDiffusionPipeline
    pipe = StableDiffusionPipeline.from_pretrained(
        meta["base_model"],
        torch_dtype=torch.float16,
        safety_checker=None,
    )

pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
pipe.to("cuda")

# Save VRAM on T4 — decode large images in slices/tiles
pipe.enable_vae_slicing()
pipe.enable_vae_tiling()

# Load LoRA via PEFT (matches how train_lora.py saves)
from peft import PeftModel
pipe.unet = PeftModel.from_pretrained(pipe.unet, str(lora_dir))
pipe.unet.eval()
print("LoRA loaded via PEFT.")

# Generate
print(f"\nGenerating {NUM_IMAGES} images at {IMAGE_SIZE}x{IMAGE_SIZE}…")
images = []
for i in range(NUM_IMAGES):
    with torch.autocast("cuda"):
        result = pipe(
            TEST_PROMPT,
            negative_prompt=NEGATIVE_PROMPT,
            height=IMAGE_SIZE,
            width=IMAGE_SIZE,
            num_inference_steps=INFERENCE_STEPS,
            guidance_scale=GUIDANCE_SCALE,
            generator=torch.Generator("cuda").manual_seed(42 + i),
        )
    images.append(result.images[0])

# Display
cols = min(NUM_IMAGES, 4)
rows = (NUM_IMAGES + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
if NUM_IMAGES == 1:
    axes = [axes]
else:
    axes = axes.flat if hasattr(axes, 'flat') else [axes]

for i, (ax, img) in enumerate(zip(axes, images)):
    ax.imshow(img)
    ax.set_title(f"Seed {42+i}")
    ax.axis("off")

# Hide empty subplots
for j in range(i + 1, len(list(axes)) if hasattr(axes, '__len__') else 0):
    axes[j].axis("off")

plt.suptitle(f"'{CHARACTER_NAME}' — consistency test", fontsize=14)
plt.tight_layout()
plt.show()

# Save test outputs
test_dir = WAREHOUSE / "outputs" / f"test_{char_id}"
test_dir.mkdir(parents=True, exist_ok=True)
for i, img in enumerate(images):
    img.save(test_dir / f"test_{i+1}.png")
print(f"\nSaved to {test_dir}")

## 6. Register Characters in AssetMemoryBank
Makes trained characters available to AnimeLoom's Director Agent.

In [ ]:
#@title 6. Register Characters in Memory Bank
#@markdown Registers all tagged characters and links their trained LoRAs.

import os, json
from pathlib import Path
from director.memory_bank import AssetMemoryBank

WAREHOUSE = Path(os.environ["AI_CACHE_ROOT"])
memory = AssetMemoryBank(str(WAREHOUSE))

tagged_dir = WAREHOUSE / "datasets" / "tagged"
lora_root = WAREHOUSE / "lora"

if not tagged_dir.exists():
    print("No tagged datasets. Run step 3 first.")
else:
    for char_dir in sorted(tagged_dir.iterdir()):
        if not char_dir.is_dir():
            continue

        name = char_dir.name
        images = sorted(
            str(f) for f in char_dir.iterdir()
            if f.suffix.lower() in {".png", ".jpg", ".jpeg", ".webp"}
        )
        if not images:
            continue

        # Read description from first caption
        desc = ""
        txt = next(char_dir.glob("*.txt"), None)
        if txt:
            desc = txt.read_text().strip()

        # Create or skip
        existing = memory.get_character(name)
        if existing:
            print(f"  '{name}' already registered")
        else:
            cid = memory.create_character(name=name, images=images, description=desc)
            print(f"  Registered '{name}' ({len(images)} images) -> {cid}")

        # Link LoRA if trained (PEFT saves as adapter_model.safetensors)
        lora_dir = lora_root / name
        meta_path = lora_dir / "metadata.json"
        if meta_path.exists():
            # Find the actual weight file
            adapter_path = lora_dir / "adapter_model.safetensors"
            legacy_path = lora_dir / "pytorch_lora_weights.safetensors"
            if adapter_path.exists():
                lora_path = str(adapter_path)
            elif legacy_path.exists():
                lora_path = str(legacy_path)
            else:
                lora_path = str(adapter_path)  # store expected path
            memory.update_character_lora(name, lora_path)
            print(f"    Linked LoRA: {lora_path}")

    memory.save_checkpoint()
    print(f"\n✅ Memory bank saved. {len(memory.list_characters())} characters registered.")

## 7. Dashboard & Export

In [ ]:
#@title 7a. Training Dashboard
#@markdown View all trained characters and their stats.

import os, json
from pathlib import Path
import matplotlib.pyplot as plt

WAREHOUSE = Path(os.environ["AI_CACHE_ROOT"])
lora_root = WAREHOUSE / "lora"

if not lora_root.exists():
    print("No trained LoRAs yet.")
else:
    characters = [d for d in sorted(lora_root.iterdir()) if d.is_dir()]
    if not characters:
        print("No trained LoRAs yet.")
    else:
        names, n_images, sizes_mb = [], [], []
        print(f"{'Character':<20} {'Images':>7} {'Steps':>7} {'Rank':>5} {'Size':>8} {'Model'}")
        print("-" * 80)

        for char_dir in characters:
            meta_path = char_dir / "metadata.json"
            if not meta_path.exists():
                continue
            m = json.loads(meta_path.read_text())

            # PEFT saves as adapter_model.safetensors, fallback to pytorch_lora_weights.safetensors
            weight_file = char_dir / "adapter_model.safetensors"
            if not weight_file.exists():
                weight_file = char_dir / "pytorch_lora_weights.safetensors"
            size = weight_file.stat().st_size / 1e6 if weight_file.exists() else 0

            names.append(m.get("character_name", char_dir.name))
            n_images.append(m.get("num_images", 0))
            sizes_mb.append(round(size, 1))

            print(
                f"{m.get('character_name', '?'):<20} "
                f"{m.get('num_images', '?'):>7} "
                f"{m.get('training_steps', '?'):>7} "
                f"{m.get('rank', '?'):>5} "
                f"{size:>7.1f}M "
                f"{m.get('base_model', '?').split('/')[-1]}"
            )

        if names:
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3))
            ax1.barh(names, n_images, color="#4e79a7")
            ax1.set_xlabel("Training Images")
            ax1.set_title("Images per Character")
            ax2.barh(names, sizes_mb, color="#e15759")
            ax2.set_xlabel("Size (MB)")
            ax2.set_title("LoRA File Size")
            plt.tight_layout()
            plt.show()

In [ ]:
#@title 7b. Export All LoRAs as ZIP
#@markdown Creates a downloadable zip of all trained character LoRAs.

import os, json, zipfile
from pathlib import Path
from datetime import datetime

WAREHOUSE = Path(os.environ["AI_CACHE_ROOT"])
lora_root = WAREHOUSE / "lora"
export_dir = WAREHOUSE / "exports"
export_dir.mkdir(parents=True, exist_ok=True)

characters = [d for d in sorted(lora_root.iterdir()) if d.is_dir()] if lora_root.exists() else []

if not characters:
    print("No trained LoRAs to export.")
else:
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    zip_path = export_dir / f"aniloom_loras_{ts}.zip"

    manifest = {"exported": datetime.now().isoformat(), "characters": []}

    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for char_dir in characters:
            meta_path = char_dir / "metadata.json"
            if meta_path.exists():
                manifest["characters"].append(json.loads(meta_path.read_text()))

            for f in char_dir.iterdir():
                zf.write(f, f"lora/{char_dir.name}/{f.name}")
                print(f"  Added {char_dir.name}/{f.name}")

        # Write manifest inside zip
        zf.writestr("manifest.json", json.dumps(manifest, indent=2))

    size_mb = zip_path.stat().st_size / 1e6
    print(f"\n✅ Export: {zip_path}  ({size_mb:.1f} MB)")
    print(f"   {len(manifest['characters'])} characters packaged.")
    print("\nTo download: right-click the file in the Drive sidebar → Download")

## 8. Colab Survival Mode
Keep the session alive during long training runs.

In [ ]:
#@title 8. Enable Colab Survival (optional)
#@markdown Prevents Colab from disconnecting during long training.
#@markdown Checkpoints every 5 minutes to Drive.

import os, sys
sys.path.insert(0, "/content/AnimeLoom")

from cloud.colab_survival import ColabSurvival

survival = ColabSurvival(
    warehouse_path=os.environ["AI_CACHE_ROOT"],
    keepalive_interval=240,   # ping every 4 min
    checkpoint_interval=300,  # save every 5 min
)
survival.start()
print("Survival daemon running — keepalive 4min, checkpoint 5min.")
print("Your work is auto-saved to Google Drive.")

## 9. Anime Video Generation (SDXL + CogVideoX)
Generates anime-style video with **real motion** — smiling, expressions, head movement, hair swaying.

| Stage | What | Purpose |
|-------|------|---------|
| **SDXL + LoRA** | High-res keyframes | Character identity locked in via your trained LoRA |
| **CogVideoX-5B** | Image-to-video | Turns each keyframe into 49 frames of actual anime motion |
| **Stitch** | Cross-fade + assemble | Joins motion clips into a continuous video |

Each model loads/unloads sequentially to fit on T4 (16GB VRAM). CogVideoX uses int8 quantization to stay under 8GB.

In [ ]:
#@title 9. Generate Anime Video (SDXL + CogVideoX)
#@markdown Real anime motion — smiling, expressions, movement.
#@markdown Uses SDXL LoRA for character identity + CogVideoX for anime-style video.

CHARACTER_NAME = "yuki_nagato"  #@param {type:"string"}
NEGATIVE_PROMPT = "blurry, low quality, deformed, extra fingers, worst quality, ugly, disfigured, bad anatomy, bad hands"  #@param {type:"string"}
DENOISING_STRENGTH = 0.55  #@param {type:"slider", min:0.3, max:0.8, step:0.05}
IMAGE_SIZE = 768  #@param {type:"slider", min:512, max:1024, step:128}
LORA_SCALE = 1.0  #@param {type:"slider", min:0.5, max:2.0, step:0.1}
SDXL_GUIDANCE = 7.5  #@param {type:"slider", min:1.0, max:15.0, step:0.5}
SDXL_STEPS = 30  #@param {type:"slider", min:10, max:50, step:5}

#@markdown ---
#@markdown **CogVideoX Motion Settings**
NUM_FRAMES = 49  #@param {type:"slider", min:13, max:49, step:12}
COGVID_STEPS = 30  #@param {type:"slider", min:20, max:50, step:5}
COGVID_GUIDANCE = 6.0  #@param {type:"slider", min:1.0, max:12.0, step:0.5}
USE_DYNAMIC_CFG = True  #@param {type:"boolean"}
FPS = 12  #@param {type:"slider", min:8, max:24, step:4}

#@markdown **Keyframe prompts** — one per line. Each becomes a keyframe, then CogVideoX animates it.
KEYFRAME_PROMPTS = """yuki_nagato, 1girl, close up portrait, detailed face, looking at viewer, school uniform, anime, masterpiece, best quality
yuki_nagato, 1girl, close up portrait, detailed face, slight smile, tilting head, school uniform, anime, masterpiece, best quality
yuki_nagato, 1girl, close up portrait, detailed face, looking to the side, wind in hair, school uniform, anime, masterpiece, best quality
yuki_nagato, 1girl, close up portrait, detailed face, serious expression, school uniform, anime, masterpiece, best quality
yuki_nagato, 1girl, close up portrait, detailed face, eyes closed, peaceful, school uniform, anime, masterpiece, best quality
yuki_nagato, 1girl, close up portrait, detailed face, looking at viewer, slight smile, school uniform, anime, masterpiece, best quality"""  #@param {type:"string"}

#@markdown **Motion prompts** — describe the motion for CogVideoX. One per keyframe (cycles if fewer).
MOTION_PROMPTS = """anime girl looking at viewer, soft breathing, gentle blink, hair swaying slightly, warm lighting
anime girl smiling gently, soft head tilt, eyes sparkling, school uniform, calm atmosphere
anime girl turning head to the side, wind blowing through hair, serene expression
anime girl with serious focused expression, slight eye movement, dramatic lighting
anime girl closing eyes peacefully, gentle sigh, relaxed shoulders, soft light
anime girl opening eyes with a warm smile, looking directly at viewer, inviting expression"""  #@param {type:"string"}

import os, gc, sys, time, torch, json, subprocess
import numpy as np
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt

WAREHOUSE = Path(os.environ["AI_CACHE_ROOT"])
char_id = CHARACTER_NAME.lower().replace(" ", "_")

kf_prompts = [p.strip() for p in KEYFRAME_PROMPTS.strip().split("\n") if p.strip()]
motion_prompts = [p.strip() for p in MOTION_PROMPTS.strip().split("\n") if p.strip()]
num_kf = len(kf_prompts)

total_frames = num_kf * NUM_FRAMES
print(f"Plan: {num_kf} keyframes × {NUM_FRAMES} CogVideoX frames = {total_frames} frames ({total_frames/FPS:.1f}s at {FPS}fps)")
print("=" * 60)

# ================================================================
# Phase 1: SDXL + LoRA — Generate keyframes with character identity
# ================================================================
gc.collect()
torch.cuda.empty_cache()

lora_dir = WAREHOUSE / "lora" / char_id
meta_path = lora_dir / "metadata.json"
if not meta_path.exists():
    raise FileNotFoundError(f"No SDXL LoRA for '{CHARACTER_NAME}' at {lora_dir}. Run step 4 first.")

meta = json.loads(meta_path.read_text())
base_model = meta["base_model"]
print(f"Loading {base_model}…")

from diffusers import StableDiffusionXLPipeline, AutoPipelineForImage2Image, DPMSolverMultistepScheduler
from peft import PeftModel

pipe = StableDiffusionXLPipeline.from_pretrained(base_model, torch_dtype=torch.float16)
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
pipe.unet = PeftModel.from_pretrained(pipe.unet, str(lora_dir))
pipe.unet.eval()

if LORA_SCALE != 1.0:
    for module in pipe.unet.modules():
        if hasattr(module, "scaling"):
            for key in module.scaling:
                module.scaling[key] = LORA_SCALE

pipe.to("cuda")
pipe.enable_vae_slicing()
pipe.enable_vae_tiling()
try:
    pipe.enable_xformers_memory_efficient_attention()
except Exception:
    pass

pipe_i2i = AutoPipelineForImage2Image.from_pipe(pipe)
print(f"SDXL + LoRA loaded (scale={LORA_SCALE}).")

print(f"\nGenerating {num_kf} keyframes…")
keyframes = []

for i, prompt in enumerate(kf_prompts):
    gen = torch.Generator("cuda").manual_seed(42 + i)

    if i == 0:
        result = pipe(
            prompt=prompt,
            negative_prompt=NEGATIVE_PROMPT,
            height=IMAGE_SIZE, width=IMAGE_SIZE,
            num_inference_steps=SDXL_STEPS,
            guidance_scale=SDXL_GUIDANCE,
            generator=gen,
        )
    else:
        result = pipe_i2i(
            prompt=prompt,
            negative_prompt=NEGATIVE_PROMPT,
            image=keyframes[-1],
            strength=DENOISING_STRENGTH,
            num_inference_steps=SDXL_STEPS,
            guidance_scale=SDXL_GUIDANCE,
            generator=gen,
        )

    keyframes.append(result.images[0])
    print(f"  Keyframe {i+1}/{num_kf} done")

kf_dir = WAREHOUSE / "outputs" / f"keyframes_{char_id}"
kf_dir.mkdir(parents=True, exist_ok=True)
for i, kf in enumerate(keyframes):
    kf.save(kf_dir / f"kf_{i:03d}.png")
print(f"Keyframes saved to {kf_dir}")

# Unload SDXL
while hasattr(pipe.unet, "base_model"):
    try:
        pipe.unet = pipe.unet.base_model.model
    except Exception:
        break
del pipe, pipe_i2i
gc.collect()
torch.cuda.empty_cache()
print("SDXL unloaded.\n")

# ================================================================
# Phase 2: CogVideoX — Animate each keyframe into real motion
# ================================================================
print("Loading CogVideoX-5B image-to-video (int8 quantized)…")

from diffusers import CogVideoXImageToVideoPipeline
from optimum.quanto import qint8, quantize, freeze

cogvid_pipe = CogVideoXImageToVideoPipeline.from_pretrained(
    "THUDM/CogVideoX-5b-I2V",
    torch_dtype=torch.bfloat16,
)

# Int8 quantization — critical for T4 VRAM
quantize(cogvid_pipe.transformer, weights=qint8)
freeze(cogvid_pipe.transformer)
quantize(cogvid_pipe.text_encoder, weights=qint8)
freeze(cogvid_pipe.text_encoder)

cogvid_pipe.enable_model_cpu_offload()
cogvid_pipe.vae.enable_tiling()
cogvid_pipe.vae.enable_slicing()
print("CogVideoX loaded and quantized.")

# Generate motion clips
all_clips = []
start_time = time.time()

for i, kf_image in enumerate(keyframes):
    motion_prompt = motion_prompts[i % len(motion_prompts)]
    print(f"\n  Animating keyframe {i+1}/{num_kf}: \"{motion_prompt[:60]}…\"")

    # Resize to CogVideoX-compatible dimensions (480x720)
    kf_resized = kf_image.resize((720, 480), Image.LANCZOS)

    gen = torch.Generator("cpu").manual_seed(42 + i)

    output = cogvid_pipe(
        image=kf_resized,
        prompt=motion_prompt,
        num_frames=NUM_FRAMES,
        num_inference_steps=COGVID_STEPS,
        guidance_scale=COGVID_GUIDANCE,
        use_dynamic_cfg=USE_DYNAMIC_CFG,
        generator=gen,
    )

    clip_frames = output.frames[0]  # list of PIL Images
    all_clips.append(clip_frames)

    elapsed = time.time() - start_time
    eta = (elapsed / (i + 1)) * (num_kf - i - 1)
    print(f"  Clip {i+1}/{num_kf} done — {len(clip_frames)} frames ({elapsed:.0f}s elapsed, ~{eta:.0f}s remaining)")

    gc.collect()
    torch.cuda.empty_cache()

cogvid_time = time.time() - start_time
print(f"\nCogVideoX done in {cogvid_time/60:.1f} minutes.")

# Unload CogVideoX
del cogvid_pipe
gc.collect()
torch.cuda.empty_cache()
print("CogVideoX unloaded.\n")

# ================================================================
# Phase 3: Stitch clips with cross-fade transitions + save video
# ================================================================
CROSSFADE_FRAMES = 4  # smooth transition between clips

print(f"Stitching {len(all_clips)} clips (cross-fade={CROSSFADE_FRAMES} frames)…")

all_frames = []
for clip_idx, clip in enumerate(all_clips):
    clip_arrays = [np.array(frame) for frame in clip]

    if clip_idx == 0:
        all_frames.extend(clip_arrays)
    else:
        # Cross-fade between last frames of previous clip and first frames of this clip
        overlap = min(CROSSFADE_FRAMES, len(all_frames), len(clip_arrays))
        for j in range(overlap):
            t = (j + 1) / (overlap + 1)
            prev_frame = all_frames[-(overlap - j)].astype(np.float32)
            next_frame = clip_arrays[j].astype(np.float32)
            blended = ((1 - t) * prev_frame + t * next_frame).clip(0, 255).astype(np.uint8)
            all_frames[-(overlap - j)] = blended
        # Append remaining frames after the overlap
        all_frames.extend(clip_arrays[overlap:])

print(f"Total: {len(all_frames)} frames ({len(all_frames)/FPS:.1f}s at {FPS}fps)")

# Save video
output_dir = WAREHOUSE / "outputs" / f"cogvid_{char_id}"
output_dir.mkdir(parents=True, exist_ok=True)
video_path = output_dir / f"{char_id}_clip.mp4"

import cv2
h, w = all_frames[0].shape[:2]
writer = cv2.VideoWriter(str(video_path), cv2.VideoWriter_fourcc(*"mp4v"), FPS, (w, h))
for frame in all_frames:
    writer.write(cv2.cvtColor(frame, cv2.COLOR_RGB2BGR))
writer.release()

duration = len(all_frames) / FPS
size_mb = video_path.stat().st_size / 1e6
total_time = time.time() - start_time
print(f"\nVideo saved: {video_path}")
print(f"Duration: {duration:.1f}s | Frames: {len(all_frames)} | Size: {size_mb:.1f}MB")
print(f"Total time: {total_time/60:.1f} minutes")

# Show keyframe preview
cols = min(num_kf, 6)
rows = (num_kf + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(3 * cols, 3 * rows))
axes_flat = np.array(axes).flat if hasattr(axes, 'flat') else [axes]
for i, (ax, kf) in enumerate(zip(axes_flat, keyframes)):
    ax.imshow(kf)
    ax.set_title(f"KF {i}", fontsize=9)
    ax.axis("off")
for j in range(i + 1, rows * cols):
    axes_flat[j].axis("off")
plt.suptitle(f"'{CHARACTER_NAME}' — SDXL keyframes (animated by CogVideoX)", fontsize=13)
plt.tight_layout()
plt.show()

gc.collect()
torch.cuda.empty_cache()
print("\nDone. VRAM freed.")

## 10. Generate 2-Minute Long Anime Video
Same CogVideoX pipeline as step 9, but scaled up. Generates many SDXL keyframes and animates each into a motion clip, then stitches everything into a long continuous anime video. Run step 9 first to verify quality.

In [ ]:
#@title 10. Generate 2-Minute Long Anime Video (SDXL + CogVideoX)
#@markdown Generates a long anime video with real motion. Run step 9 first to verify quality.
#@markdown ~60-120 min on T4 depending on target duration.

CHARACTER_NAME = "yuki_nagato"  #@param {type:"string"}
TARGET_SECONDS = 120  #@param {type:"slider", min:30, max:300, step:30}
FPS = 12  #@param {type:"slider", min:8, max:24, step:4}
DENOISING_STRENGTH = 0.50  #@param {type:"slider", min:0.3, max:0.8, step:0.05}
IMAGE_SIZE = 768  #@param {type:"slider", min:512, max:1024, step:128}
LORA_SCALE = 1.0  #@param {type:"slider", min:0.5, max:2.0, step:0.1}
SDXL_GUIDANCE = 7.5  #@param {type:"slider", min:1.0, max:15.0, step:0.5}
SDXL_STEPS = 25  #@param {type:"slider", min:10, max:50, step:5}

#@markdown ---
#@markdown **CogVideoX Settings**
NUM_FRAMES = 49  #@param {type:"slider", min:13, max:49, step:12}
COGVID_STEPS = 30  #@param {type:"slider", min:20, max:50, step:5}
COGVID_GUIDANCE = 6.0  #@param {type:"slider", min:1.0, max:12.0, step:0.5}
USE_DYNAMIC_CFG = True  #@param {type:"boolean"}

#@markdown Scene prompts — the pipeline cycles through these for variety.
SCENE_PROMPTS = """yuki_nagato, 1girl, close up portrait, detailed face, walking through school hallway, school uniform, anime, masterpiece, best quality
yuki_nagato, 1girl, close up portrait, detailed face, sitting at desk reading book, classroom, school uniform, anime, masterpiece, best quality
yuki_nagato, 1girl, close up portrait, detailed face, standing by window, sunset light, school uniform, anime, masterpiece, best quality
yuki_nagato, 1girl, close up portrait, detailed face, walking outside, cherry blossoms, school uniform, anime, masterpiece, best quality"""  #@param {type:"string"}

#@markdown Motion prompts — describe the animation per scene.
MOTION_PROMPTS = """anime girl walking forward in school hallway, steady footsteps, looking around, hair bouncing
anime girl sitting at desk, reading a book, turning page, calm focus, eyes scanning text
anime girl standing by window, gazing outside, sunset glow on face, wind gently moving hair
anime girl walking outside under cherry blossoms, petals falling, happy expression, looking up"""  #@param {type:"string"}

NEGATIVE_PROMPT = "blurry, low quality, deformed, extra fingers, worst quality, ugly, disfigured, bad anatomy, bad hands"  #@param {type:"string"}

import os, gc, sys, time, torch, json, subprocess
import numpy as np
from pathlib import Path
from PIL import Image

WAREHOUSE = Path(os.environ["AI_CACHE_ROOT"])
char_id = CHARACTER_NAME.lower().replace(" ", "_")
scenes = [s.strip() for s in SCENE_PROMPTS.strip().split("\n") if s.strip()]
motions = [s.strip() for s in MOTION_PROMPTS.strip().split("\n") if s.strip()]

# Calculate keyframes needed
CROSSFADE_FRAMES = 4
effective_frames_per_clip = NUM_FRAMES - CROSSFADE_FRAMES
total_frames_needed = TARGET_SECONDS * FPS
num_keyframes = max(2, (total_frames_needed + effective_frames_per_clip - 1) // effective_frames_per_clip)

total_frames = num_keyframes * NUM_FRAMES - (num_keyframes - 1) * CROSSFADE_FRAMES
actual_duration = total_frames / FPS

print(f"Target: {TARGET_SECONDS}s at {FPS}fps")
print(f"Keyframes needed: {num_keyframes}")
print(f"After CogVideoX + stitch: ~{total_frames} frames ({actual_duration:.1f}s)")
print(f"Scene prompts: {len(scenes)} (cycling)")
print("=" * 60)

# ================================================================
# Phase 1: SDXL + LoRA — Generate keyframes
# ================================================================
gc.collect()
torch.cuda.empty_cache()

lora_dir = WAREHOUSE / "lora" / char_id
meta_path = lora_dir / "metadata.json"
if not meta_path.exists():
    raise FileNotFoundError(f"No SDXL LoRA for '{CHARACTER_NAME}'. Run step 4 first.")

meta = json.loads(meta_path.read_text())
base_model = meta["base_model"]
print(f"\nLoading {base_model}…")

from diffusers import StableDiffusionXLPipeline, AutoPipelineForImage2Image, DPMSolverMultistepScheduler
from peft import PeftModel

pipe = StableDiffusionXLPipeline.from_pretrained(base_model, torch_dtype=torch.float16)
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
pipe.unet = PeftModel.from_pretrained(pipe.unet, str(lora_dir))
pipe.unet.eval()

if LORA_SCALE != 1.0:
    for module in pipe.unet.modules():
        if hasattr(module, "scaling"):
            for key in module.scaling:
                module.scaling[key] = LORA_SCALE

pipe.to("cuda")
pipe.enable_vae_slicing()
pipe.enable_vae_tiling()
try:
    pipe.enable_xformers_memory_efficient_attention()
except Exception:
    pass

pipe_i2i = AutoPipelineForImage2Image.from_pipe(pipe)
print("SDXL + LoRA loaded.")

print(f"\nGenerating {num_keyframes} keyframes…")
keyframes = []
kf_start = time.time()

for i in range(num_keyframes):
    prompt = scenes[i % len(scenes)]
    gen = torch.Generator("cuda").manual_seed(42 + i * 3)

    if i == 0:
        result = pipe(
            prompt=prompt,
            negative_prompt=NEGATIVE_PROMPT,
            height=IMAGE_SIZE, width=IMAGE_SIZE,
            num_inference_steps=SDXL_STEPS,
            guidance_scale=SDXL_GUIDANCE,
            generator=gen,
        )
    else:
        result = pipe_i2i(
            prompt=prompt,
            negative_prompt=NEGATIVE_PROMPT,
            image=keyframes[-1],
            strength=DENOISING_STRENGTH,
            num_inference_steps=SDXL_STEPS,
            guidance_scale=SDXL_GUIDANCE,
            generator=gen,
        )

    keyframes.append(result.images[0])

    elapsed = time.time() - kf_start
    eta = (elapsed / (i + 1)) * (num_keyframes - i - 1)
    print(f"  KF {i+1}/{num_keyframes} done ({elapsed:.0f}s elapsed, ~{eta:.0f}s remaining)")

    if i % 20 == 19:
        gc.collect()
        torch.cuda.empty_cache()

kf_time = time.time() - kf_start
print(f"Keyframes generated in {kf_time/60:.1f} minutes.")

# Save keyframes
kf_dir = WAREHOUSE / "outputs" / f"keyframes_{char_id}_long"
kf_dir.mkdir(parents=True, exist_ok=True)
for i, kf in enumerate(keyframes):
    kf.save(kf_dir / f"kf_{i:04d}.png")

# Unload SDXL
while hasattr(pipe.unet, "base_model"):
    try:
        pipe.unet = pipe.unet.base_model.model
    except Exception:
        break
del pipe, pipe_i2i
gc.collect()
torch.cuda.empty_cache()
print("SDXL unloaded.\n")

# ================================================================
# Phase 2: CogVideoX — Animate each keyframe
# ================================================================
print("Loading CogVideoX-5B image-to-video (int8 quantized)…")

from diffusers import CogVideoXImageToVideoPipeline
from optimum.quanto import qint8, quantize, freeze

cogvid_pipe = CogVideoXImageToVideoPipeline.from_pretrained(
    "THUDM/CogVideoX-5b-I2V",
    torch_dtype=torch.bfloat16,
)

quantize(cogvid_pipe.transformer, weights=qint8)
freeze(cogvid_pipe.transformer)
quantize(cogvid_pipe.text_encoder, weights=qint8)
freeze(cogvid_pipe.text_encoder)

cogvid_pipe.enable_model_cpu_offload()
cogvid_pipe.vae.enable_tiling()
cogvid_pipe.vae.enable_slicing()
print("CogVideoX loaded and quantized.")

# Process keyframes and write frames to disk to save RAM
frames_dir = WAREHOUSE / "outputs" / f"frames_{char_id}_long"
frames_dir.mkdir(parents=True, exist_ok=True)

frame_count = 0
cogvid_start = time.time()

# We'll track last few frames of previous clip for cross-fade
prev_tail_frames = None

for i, kf_image in enumerate(keyframes):
    motion_prompt = motions[i % len(motions)]
    print(f"\n  Animating KF {i+1}/{num_keyframes}: \"{motion_prompt[:50]}…\"")

    kf_resized = kf_image.resize((720, 480), Image.LANCZOS)
    gen = torch.Generator("cpu").manual_seed(42 + i)

    output = cogvid_pipe(
        image=kf_resized,
        prompt=motion_prompt,
        num_frames=NUM_FRAMES,
        num_inference_steps=COGVID_STEPS,
        guidance_scale=COGVID_GUIDANCE,
        use_dynamic_cfg=USE_DYNAMIC_CFG,
        generator=gen,
    )

    clip_frames = [np.array(f) for f in output.frames[0]]

    if prev_tail_frames is not None:
        # Cross-fade transition
        overlap = min(CROSSFADE_FRAMES, len(prev_tail_frames), len(clip_frames))
        for j in range(overlap):
            t = (j + 1) / (overlap + 1)
            prev_f = prev_tail_frames[-(overlap - j)].astype(np.float32)
            next_f = clip_frames[j].astype(np.float32)
            blended = ((1 - t) * prev_f + t * next_f).clip(0, 255).astype(np.uint8)
            # Overwrite the previously saved tail frames
            overwrite_idx = frame_count - (overlap - j)
            Image.fromarray(blended).save(frames_dir / f"frame_{overwrite_idx:06d}.png")
        # Save new frames after overlap
        for f in clip_frames[overlap:]:
            Image.fromarray(f).save(frames_dir / f"frame_{frame_count:06d}.png")
            frame_count += 1
    else:
        # First clip — save all frames
        for f in clip_frames:
            Image.fromarray(f).save(frames_dir / f"frame_{frame_count:06d}.png")
            frame_count += 1

    # Keep tail frames for next cross-fade
    prev_tail_frames = clip_frames[-CROSSFADE_FRAMES:]

    elapsed = time.time() - cogvid_start
    eta = (elapsed / (i + 1)) * (num_keyframes - i - 1)
    print(f"  Clip {i+1}/{num_keyframes} done — {len(clip_frames)} frames ({elapsed:.0f}s elapsed, ~{eta/60:.0f}m remaining)")

    gc.collect()
    torch.cuda.empty_cache()

cogvid_time = time.time() - cogvid_start
print(f"\nCogVideoX done in {cogvid_time/60:.1f} minutes. {frame_count} frames saved to disk.")

del cogvid_pipe
gc.collect()
torch.cuda.empty_cache()

# ================================================================
# Phase 3: Assemble video from saved frames
# ================================================================
print(f"\nAssembling video from {frame_count} frames…")

import cv2

output_dir = WAREHOUSE / "outputs" / "long_video"
output_dir.mkdir(parents=True, exist_ok=True)
video_path = output_dir / f"{char_id}_{TARGET_SECONDS}s.mp4"

# Read first frame to get dimensions
first_frame = np.array(Image.open(frames_dir / "frame_000000.png"))
h, w = first_frame.shape[:2]

writer = cv2.VideoWriter(str(video_path), cv2.VideoWriter_fourcc(*"mp4v"), FPS, (w, h))
for idx in range(frame_count):
    frame_path = frames_dir / f"frame_{idx:06d}.png"
    if frame_path.exists():
        frame = np.array(Image.open(frame_path))
        writer.write(cv2.cvtColor(frame, cv2.COLOR_RGB2BGR))
writer.release()

duration = frame_count / FPS
size_mb = video_path.stat().st_size / 1e6
total_time = time.time() - kf_start

print(f"\nVideo saved: {video_path}")
print(f"Duration: {duration:.1f}s | Frames: {frame_count} | Size: {size_mb:.1f}MB")
print(f"Total time: {total_time/60:.1f} minutes (keyframes: {kf_time/60:.1f}m, CogVideoX: {cogvid_time/60:.1f}m)")

gc.collect()
torch.cuda.empty_cache()
print("Done. VRAM freed.")

---

## Quick Reference

| Step | What | Time (Free/Pro) |
|------|------|------------------|
| 1 | Setup | 2 min |
| 2 | Get images | 1–5 min |
| 3 | Caption | 5–15 min |
| 4 | Train SDXL LoRA | 30–60 min / 15–30 min |
| 4c | Train SD 1.5 LoRA (optional) | 20–40 min / 10–20 min |
| 5 | Test | 1–2 min |
| 6 | Register | < 1 min |
| 7 | Dashboard + Export | < 1 min |
| 9 | **Anime video clip (CogVideoX)** | **60–90 min** |
| 10 | **2-min long anime video** | **90–120 min** |

### After training
Your LoRAs persist on Google Drive at:
```
MyDrive/AniLoom/warehouse/lora/
├── {character_name}/                    # SDXL LoRA
│   ├── adapter_model.safetensors
│   ├── adapter_config.json
│   └── metadata.json
└── {character_name}_sd15/               # SD 1.5 LoRA (optional, for AnimateDiff)
    ├── adapter_model.safetensors
    ├── adapter_config.json
    └── metadata.json
```

### Pipeline order for full animation
1. Train SDXL LoRA (step 4) — character identity
2. Register characters (step 6)
3. **Test anime clip (step 9)** — SDXL keyframes + CogVideoX motion
4. **Generate long anime video (step 10)** — 2+ minutes

### Key parameters for video quality
- **DENOISING_STRENGTH** (0.3–0.8): Lower = more consistent face between keyframes, higher = more scene variety
- **COGVID_GUIDANCE** (1.0–12.0): Higher = follows motion prompt more strictly
- **COGVID_STEPS** (20–50): More steps = better quality but slower
- **NUM_FRAMES** (13–49): Frames per CogVideoX clip. 49 = ~4s of motion per keyframe
- **LORA_SCALE** (0.5–2.0): Boost if character isn't recognizable enough
- **Motion prompts**: Describe the action you want — "smiling", "head tilt", "wind in hair", etc.

To use in AnimeLoom locally:
```bash
export AI_CACHE_ROOT=./warehouse
python main.py --script my_story.txt
```